# 📦 Model prefetch to Google Drive (bypass a flaky local network)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/model_prefetch_to_drive.ipynb)

Downloads every model this project's local benchmarks need using Colab's network (fast, stable) instead of a local machine's -- useful when the local network (e.g. a VPN reporting `Transient Connection`) keeps stalling multi-gigabyte Hugging Face downloads. Packs each model into a single `.tar` per repo (preserving the Hugging Face cache's internal blob/snapshot symlink layout -- Drive's FUSE mount does not reliably preserve symlinks on its own, so they must travel inside an archive, not as loose files) and copies it to Google Drive.

**No GPU needed** -- this notebook only downloads and packs files, so pick a CPU-only runtime to avoid burning GPU quota on a pure I/O task.

## After this notebook finishes

On the local machine, for each archive downloaded from `MyDrive/higgs-benchmark/model-cache/`:

```bash
tar -xf ~/Downloads/<name>.tar -C ~/.cache/huggingface/hub/
```

That's it -- `huggingface_hub` recognizes the extracted `models--<org>--<name>` directory as already cached (same revision, same blob hashes) and every later `make tts` / `make stt` / Qwen local run skips downloading it again.


## 1. Google Drive workspace

Same pattern as the other notebooks in this repo: results (here, the archives) go to Drive as each one finishes, and the VM disconnects unconditionally at the end.


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

MODEL_CACHE_DIR = Path("/content/drive/MyDrive/higgs-benchmark/model-cache")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Archives will be written to: {MODEL_CACHE_DIR}")


## 2. Choose which models to fetch

Uncomment/comment entries to skip models you already have locally (e.g. Higgs TTS 3 if a local download already got far enough). Each entry is `(archive_name, repo_id, revision_or_None, hf_cache_env)` -- `hf_cache_env` picks the right cache dir prefix (`hub` for `huggingface_hub.snapshot_download`, matching `~/.cache/huggingface/hub` on the local machine for every model here, MLX included, since MLX-Audio also stores weights through `huggingface_hub`).


In [ ]:
MODELS = [
    ("higgs-tts-3-4b", "bosonai/higgs-tts-3-4b", None),
    ("higgs-audio-v3-stt", "bosonai/higgs-audio-v3-stt", "2ffd1aa39f5a1266931e405cba12e404a9f994b2"),
    ("whisper-large-v3", "openai/whisper-large-v3", None),
    ("qwen3-tts-0.6b-base", "mlx-community/Qwen3-TTS-12Hz-0.6B-Base-bf16", None),
    ("qwen3-tts-1.7b-customvoice", "mlx-community/Qwen3-TTS-12Hz-1.7B-CustomVoice-bf16", None),
    ("qwen3-asr-0.6b", "mlx-community/Qwen3-ASR-0.6B-8bit", None),
]
print(f"{len(MODELS)} models queued:")
for name, repo_id, revision in MODELS:
    print(f"  - {name}: {repo_id}" + (f"@{revision}" if revision else ""))


## 3. Download each model, pack it, copy to Drive

Downloads into the VM's local disk (`HF_HOME` below) -- fast, and Colab's own network is what we're relying on being stable, not FUSE-backed Drive I/O for the download itself. Only the final archive crosses onto Drive.

Each model is archived and uploaded **immediately after it finishes**, not batched at the end: if the notebook is interrupted partway through the model list, everything already done is already safe on Drive.


In [ ]:
import os
import subprocess
import tarfile
import time
from pathlib import Path

HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_DISABLE_XET"] = "1"  # matches scripts/download_models.sh

from huggingface_hub import snapshot_download


def download_with_retry(repo_id, revision, max_attempts=10):
    for attempt in range(1, max_attempts + 1):
        try:
            return snapshot_download(repo_id, revision=revision)
        except Exception as error:
            if attempt == max_attempts:
                raise
            backoff = min(attempt * 5, 30)
            print(f"  attempt {attempt}/{max_attempts} failed ({error!r}); retrying in {backoff}s")
            time.sleep(backoff)


results = []
for name, repo_id, revision in MODELS:
    print(f"\n=== {name} ({repo_id}) ===")
    archive_path = MODEL_CACHE_DIR / f"{name}.tar"
    if archive_path.exists():
        print(f"already on Drive: {archive_path} ({archive_path.stat().st_size / 1e9:.2f} GB) -- skipping")
        results.append((name, "SKIPPED (already on Drive)", archive_path))
        continue
    try:
        download_with_retry(repo_id, revision)
        safe_repo = repo_id.replace("/", "--")
        model_dir = HF_HOME / "hub" / f"models--{safe_repo}"
        assert model_dir.is_dir(), f"expected cache dir missing: {model_dir}"

        local_tar = Path("/content") / f"{name}.tar"
        with tarfile.open(local_tar, "w") as tf:
            tf.add(model_dir, arcname=model_dir.name)
        print(f"packed {local_tar} ({local_tar.stat().st_size / 1e9:.2f} GB)")

        subprocess.run(["cp", str(local_tar), str(archive_path)], check=True)
        local_tar.unlink()  # free VM disk once the Drive copy exists
        print(f"uploaded to {archive_path}")
        results.append((name, "OK", archive_path))
    except Exception as error:
        print(f"FAILED: {error!r}")
        results.append((name, f"FAILED: {error!r}", None))

print("\n=== Summary ===")
for name, status, path in results:
    print(f"{name}: {status}" + (f" -> {path}" if path else ""))


## 4. Instructions for the local machine

Download each `.tar` from `MyDrive/higgs-benchmark/model-cache/` (Drive web UI, or let Drive desktop sync it automatically), then, in a terminal:

```bash
for f in ~/Downloads/*.tar; do
  tar -xf "$f" -C ~/.cache/huggingface/hub/
done
```

Verify with `du -sh ~/.cache/huggingface/hub/models--*` -- sizes should match what this notebook printed per model. `make tts` / `make stt` / the Qwen local runners will then find everything already cached and skip downloading.


## 5. Завершение: безусловное отключение


In [ ]:
try:
    print(f"На Диске сохранено: {MODEL_CACHE_DIR}")
    for artefact in sorted(MODEL_CACHE_DIR.iterdir()):
        if artefact.is_file():
            print(f"  - {artefact.name} ({artefact.stat().st_size / 1e9:.2f} GB)")
except Exception as error:
    print(f"не удалось перечислить артефакты: {error!r}")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("💾 Данные синхронизированы: MyDrive/higgs-benchmark/model-cache/")
finally:
    from google.colab import runtime
    print("🛑 Отключение ВМ...")
    runtime.unassign()
